# Spark Interoperability with Snowflake Iceberg V3 Tables

This notebook demonstrates querying Snowflake-managed Iceberg V3 tables from Apache Spark,
with access controls enforced via Snowflake Horizon catalog.

## Prerequisites
- Apache Spark 4.0+ (required for VARIANT support)
- Python 3.10+
- Completed the Snowflake setup from the main guide

## Cloud Storage Bundles

Even when Iceberg data lives in [Snowflake-managed storage](https://docs.snowflake.com/en/user-guide/tables-iceberg-internal-storage), Spark reads files through Iceberg’s file I/O, which uses cloud-specific SDK bundles. Match the bundle to your **Snowflake account’s cloud** (the lab defaults to the AWS bundle for AWS-hosted accounts).

| Cloud Provider | Bundle to Add |
|----------------|---------------|
| **AWS S3** | `org.apache.iceberg:iceberg-aws-bundle:1.10.1` |
| **Google Cloud** | `org.apache.iceberg:iceberg-gcp-bundle:1.10.1` |
| **Azure** | `org.apache.iceberg:iceberg-azure-bundle:1.10.1` |

Update `spark.jars.packages` in the Spark session cell if your account is on GCP or Azure.

In [1]:
import os
from dotenv import load_dotenv

# Load configuration from the same config.env used during setup
load_dotenv('config.env')

SNOWFLAKE_ACCOUNT = os.getenv('SNOWFLAKE_ACCOUNT')
SNOWFLAKE_USER = os.getenv('SNOWFLAKE_USER')
SNOWFLAKE_DATABASE = os.getenv('SNOWFLAKE_DATABASE', 'FLEET_ANALYTICS_DB')

# For Iceberg REST API, dashes must be replaced with underscores in account identifier
# IMPORTANT: Set your Snowflake account identifier for the REST API endpoint
# Use one of these formats (see https://docs.snowflake.com/en/user-guide/admin-account-identifier):
#   - orgname-accountname (e.g., "myorg-myaccount")
#   - account_locator.region.cloud (e.g., "xy12345.us-west-2.aws")
# To find yours, run in Snowflake: SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME();
SNOWFLAKE_ACCOUNT_URL = os.getenv('SNOWFLAKE_ACCOUNT_URL', SNOWFLAKE_ACCOUNT)
SNOWFLAKE_ACCOUNTADMIN_TOKEN = os.getenv('SNOWFLAKE_ACCOUNTADMIN_TOKEN')
SNOWFLAKE_FLEET_ANALYST_TOKEN = os.getenv('SNOWFLAKE_FLEET_ANALYST_TOKEN')
SCALA_VERSION = '2.13'
ICEBERG_VERSION = '1.10.1'

print(f"Snowflake Account (REST endpoint): {SNOWFLAKE_ACCOUNT_URL}")
print(f"Database: {SNOWFLAKE_DATABASE}")

Snowflake Account (REST endpoint): BR51643.ca-central-1.aws
Database: FLEET_ANALYTICS_DB


## Create Spark Session with Horizon Catalog

In [2]:
from pyspark.sql import SparkSession

# Configuration
SNOWFLAKE_PASSWORD = os.getenv('SNOWFLAKE_PASSWORD')
SNOWFLAKE_ROLE = 'ACCOUNTADMIN'
SNOWFLAKE_SCHEMA = 'RAW'
SNOWFLAKE_WAREHOUSE = os.getenv('SNOWFLAKE_WAREHOUSE', 'FLEET_ANALYTICS_WH')
SF_URL = f"{SNOWFLAKE_ACCOUNT_URL}.snowflakecomputing.com"

# Versions - Note: Spark 3.5 required for Snowflake Connector masking support
SNOWFLAKE_JDBC_VERSION = "3.24.0"
SNOWFLAKE_SPARK_CONNECTOR_VERSION = "3.1.6"

# Create Spark session with Iceberg and Snowflake catalog configuration
# Note: 
# - driver.host and bindAddress ensure Spark uses localhost (avoids VPN issues)
# - Using Spark 4.0 + Iceberg 1.10.1 for native VARIANT support
spark = SparkSession.builder \
    .appName("Fleet Analytics - Iceberg V3 Interop") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.jars.packages", 
            f"org.apache.iceberg:iceberg-spark-runtime-4.0_{SCALA_VERSION}:{ICEBERG_VERSION},"
            f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION},"
            f"net.snowflake:snowflake-jdbc:{SNOWFLAKE_JDBC_VERSION},"
            f"net.snowflake:spark-snowflake_{SCALA_VERSION}:{SNOWFLAKE_SPARK_CONNECTOR_VERSION}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.horizon", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.horizon.type", "rest") \
    .config("spark.sql.catalog.horizon.uri", f"https://{SNOWFLAKE_ACCOUNT_URL}.snowflakecomputing.com/polaris/api/catalog") \
    .config("spark.sql.catalog.horizon.credential", SNOWFLAKE_ACCOUNTADMIN_TOKEN) \
    .config("spark.sql.catalog.horizon.warehouse", SNOWFLAKE_DATABASE) \
    .config("spark.sql.catalog.horizon.scope", f"session:role:{SNOWFLAKE_ROLE}") \
    .config("spark.sql.catalog.horizon.header.X-Iceberg-Access-Delegation","vended-credentials") \
    .config("spark.snowflake.sfURL", SF_URL) \
    .config("spark.snowflake.sfUser", os.getenv('SNOWFLAKE_USER')) \
    .config("spark.snowflake.sfPassword", SNOWFLAKE_FLEET_ANALYST_TOKEN) \
    .config("spark.snowflake.sfDatabase", SNOWFLAKE_DATABASE) \
    .config("spark.snowflake.sfSchema", SNOWFLAKE_SCHEMA) \
    .config("spark.snowflake.sfRole", SNOWFLAKE_ROLE) \
    .config("spark.snowflake.sfWarehouse", SNOWFLAKE_WAREHOUSE) \
    .config("spark.sql.iceberg.vectorization.enabled", "false") \
    .getOrCreate()

print("Spark session created successfully!")
print(f"Spark version: {spark.version}")

:: loading settings :: url = jar:file:/opt/anaconda3/envs/fleet-spark/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/steal/.ivy2.5.2/cache
The jars for the packages stored in: /Users/steal/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
net.snowflake#snowflake-jdbc added as a dependency
net.snowflake#spark-snowflake_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c2e6937a-c182-4d52-996d-77133c03e447;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.10.1 in central
	found net.snowflake#snowflake-jdbc;3.24.0 in central
	found net.snowflake#spark-snowflake_2.13;3.1.6 in central
	found net.snowflake#snowflake-jdbc;3.24.2 in central
	found org.apache.parquet#parquet-avro

Spark session created successfully!
Spark version: 4.0.0


## List Available Tables (including Dynamic Tables)

In [3]:
# Show all tables visible to Spark
spark.sql("SHOW TABLES IN horizon.RAW").show(truncate=False)
spark.sql("SHOW TABLES IN horizon.CURATED").show(truncate=False)
spark.sql("SHOW TABLES IN horizon.ANALYTICS").show(truncate=False)

26/02/18 11:20:13 WARN AuthManagers: Inferring rest.auth.type=oauth2 since property credential was provided. Please explicitly set rest.auth.type to avoid this warning.
26/02/18 11:20:13 WARN OAuth2Manager: Iceberg REST client is missing the OAuth2 server URI configuration and defaults to https://BR51643.ca-central-1.aws.snowflakecomputing.com/polaris/api/catalog/v1/oauth/tokens. This automatic fallback will be removed in a future Iceberg release. It is recommended to configure the OAuth2 endpoint using the 'oauth2-server-uri' property to be prepared. This warning will disappear if the OAuth2 endpoint is explicitly configured. See https://github.com/apache/iceberg/issues/10537


+---------+------------------------+-----------+
|namespace|tableName               |isTemporary|
+---------+------------------------+-----------+
|RAW      |API_WEATHER_DATA        |false      |
|RAW      |MAINTENANCE_LOGS        |false      |
|RAW      |SENSOR_READINGS         |false      |
|RAW      |VEHICLE_LOCATIONS       |false      |
|RAW      |VEHICLE_REGISTRY        |false      |
|RAW      |VEHICLE_TELEMETRY_STREAM|false      |
+---------+------------------------+-----------+

+---------+--------------------+-----------+
|namespace|tableName           |isTemporary|
+---------+--------------------+-----------+
|CURATED  |MAINTENANCE_ANALYSIS|false      |
|CURATED  |TELEMETRY_ENRICHED  |false      |
+---------+--------------------+-----------+

+---------+--------------------+-----------+
|namespace|tableName           |isTemporary|
+---------+--------------------+-----------+
|ANALYTICS|DAILY_FLEET_SUMMARY |false      |
|ANALYTICS|VEHICLE_HEALTH_SCORE|false      |
+---------+--

In [28]:
# See variant column in Iceberg table
spark.sql("DESCRIBE TABLE horizon.CURATED.MAINTENANCE_ANALYSIS").show(truncate=False)

+-----------------------+-------------+-------+
|col_name               |data_type    |comment|
+-----------------------+-------------+-------+
|LOG_ID                 |string       |NULL   |
|VEHICLE_ID             |string       |NULL   |
|LOG_TIMESTAMP          |timestamp_ntz|NULL   |
|EVENT_TYPE             |string       |NULL   |
|SEVERITY               |string       |NULL   |
|DESCRIPTION            |string       |NULL   |
|TECHNICIAN             |string       |NULL   |
|SERVICE_CENTER         |string       |NULL   |
|PARTS_REPLACED         |variant      |NULL   |
|LABOR_HOURS            |double       |NULL   |
|TOTAL_COST             |double       |NULL   |
|DIAGNOSTIC_CODE_COUNT  |decimal(9,0) |NULL   |
|DIAGNOSTIC_CODES       |variant      |NULL   |
|NEXT_SERVICE_DATE      |date         |NULL   |
|MAKE                   |string       |NULL   |
|MODEL                  |string       |NULL   |
|YEAR                   |decimal(10,0)|NULL   |
|FLEET_REGION           |string       |N

In [29]:
spark.sql("""
SELECT _change_type, _commit_snapshot_id, _change_ordinal
FROM horizon.CURATED.MAINTENANCE_ANALYSIS.changes
LIMIT 5
""").show(truncate=False)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/anaconda3/envs/fleet-spark/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/anaconda3/envs/fleet-spark/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/anaconda3/envs/fleet-spark/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [9]:
q = "SELECT * FROM horizon.CURATED.TELEMETRY_ENRICHED.changes LIMIT 5"
df = spark.sql(q)          # should be fast if only parsing/analyzing is OK
df.explain("extended")     # if THIS hangs → analysis/planning/metadata
df.show(5, truncate=False) # if only THIS hangs → execution (scan/shuffle)

26/02/18 11:25:20 WARN S3InputStream: Unclosed input stream created by:
	org.apache.iceberg.aws.s3.S3InputStream.<init>(S3InputStream.java:102)
	org.apache.iceberg.aws.s3.S3InputFile.newStream(S3InputFile.java:167)
	org.apache.iceberg.avro.AvroIterable.newFileReader(AvroIterable.java:102)
	org.apache.iceberg.avro.AvroIterable.iterator(AvroIterable.java:77)
	org.apache.iceberg.avro.AvroIterable.iterator(AvroIterable.java:37)
	org.apache.iceberg.relocated.com.google.common.collect.Iterables.addAll(Iterables.java:332)
	org.apache.iceberg.relocated.com.google.common.collect.Lists.newLinkedList(Lists.java:261)
	org.apache.iceberg.ManifestLists.read(ManifestLists.java:42)
	org.apache.iceberg.BaseSnapshot.cacheManifests(BaseSnapshot.java:185)
	org.apache.iceberg.BaseSnapshot.deleteManifests(BaseSnapshot.java:219)
	org.apache.iceberg.BaseIncrementalChangelogScan.orderedChangelogSnapshots(BaseIncrementalChangelogScan.java:108)
	org.apache.iceberg.BaseIncrementalChangelogScan.doPlanFiles(BaseInc

KeyboardInterrupt: 

In [10]:
df = spark.sql("SELECT * FROM horizon.CURATED.TELEMETRY_ENRICHED.changes LIMIT 5")
print(df._jdf.queryExecution().optimizedPlan().toString()[:3000])

26/02/18 11:27:00 WARN S3InputStream: Unclosed input stream created by:
	org.apache.iceberg.aws.s3.S3InputStream.<init>(S3InputStream.java:102)
	org.apache.iceberg.aws.s3.S3InputFile.newStream(S3InputFile.java:167)
	org.apache.iceberg.avro.AvroIterable.newFileReader(AvroIterable.java:102)
	org.apache.iceberg.avro.AvroIterable.iterator(AvroIterable.java:77)
	org.apache.iceberg.avro.AvroIterable.iterator(AvroIterable.java:37)
	org.apache.iceberg.relocated.com.google.common.collect.Iterables.addAll(Iterables.java:332)
	org.apache.iceberg.relocated.com.google.common.collect.Lists.newLinkedList(Lists.java:261)
	org.apache.iceberg.ManifestLists.read(ManifestLists.java:42)
	org.apache.iceberg.BaseSnapshot.cacheManifests(BaseSnapshot.java:185)
	org.apache.iceberg.BaseSnapshot.deleteManifests(BaseSnapshot.java:219)
	org.apache.iceberg.BaseIncrementalChangelogScan.orderedChangelogSnapshots(BaseIncrementalChangelogScan.java:108)
	org.apache.iceberg.BaseIncrementalChangelogScan.doPlanFiles(BaseInc

GlobalLimit 5
+- LocalLimit 5
   +- RelationV2[VEHICLE_ID#313, EVENT_TIMESTAMP#314, TELEMETRY_DATA#315, SPEED_MPH#316, LATITUDE#317, LONGITUDE#318, HEADING#319, ENGINE_RPM#320, ENGINE_TEMP_F#321, OIL_PRESSURE_PSI#322, FUEL_LEVEL_PCT#323, CHECK_ENGINE_LIGHT#324, TIRE_PRESSURE_WARNING#325, HARD_ACCELERATIONS#326, HARD_BRAKES#327, SHARP_TURNS#328, MAKE#329, MODEL#330, YEAR#331, DRIVER_ID#332, DRIVER_NAME#333, FLEET_REGION#334, VEHICLE_STATUS#335, ENGINE_HEALTH_STATUS#336, DRIVING_MODE#337, ... 4 more fields] horizon.CURATED.TELEMETRY_ENRICHED.changes



In [38]:
spark.sql("""
SELECT snapshot_id, parent_id, committed_at, operation
FROM horizon.CURATED.MAINTENANCE_ANALYSIS.snapshots
WHERE committed_at < '2026-02-18'
ORDER BY committed_at DESC
""").show(truncate=False)

+-------------------+-------------------+-----------------------+---------+
|snapshot_id        |parent_id          |committed_at           |operation|
+-------------------+-------------------+-----------------------+---------+
|8055024666366074335|2174314919591714051|2026-02-17 23:57:30.468|delete   |
|2174314919591714051|4941640739959674103|2026-02-17 23:54:18.739|delete   |
|4941640739959674103|2277436848320296647|2026-02-17 23:51:05.416|delete   |
|2277436848320296647|4013801516525042891|2026-02-17 23:47:55.743|delete   |
|4013801516525042891|3305615129954053135|2026-02-17 23:44:42.618|delete   |
|3305615129954053135|2656968907227745217|2026-02-17 23:41:29.439|delete   |
|2656968907227745217|4007678449406329786|2026-02-17 23:38:18.248|delete   |
|4007678449406329786|7863955280151996971|2026-02-17 23:35:05.915|delete   |
|7863955280151996971|9154880560136007069|2026-02-17 23:31:55.213|delete   |
|9154880560136007069|6907578569922475525|2026-02-17 23:28:42.645|delete   |
|69075785699

In [39]:
start_id = "7147354370225788447"
end_id   = "2166219244961903548"

df = (spark.read
        .option("start-snapshot-id", start_id)
        .option("end-snapshot-id", end_id)
        .table("horizon.CURATED.MAINTENANCE_ANALYSIS.changes"))

df.limit(5).show(truncate=False)

+------+----------+-------------+----------+--------+-----------+----------+--------------+--------------+-----------+----------+---------------------+----------------+-----------------+----+-----+----+------------+-----------------+-----------------------+-------------+------------+---------------+-------------------+
|LOG_ID|VEHICLE_ID|LOG_TIMESTAMP|EVENT_TYPE|SEVERITY|DESCRIPTION|TECHNICIAN|SERVICE_CENTER|PARTS_REPLACED|LABOR_HOURS|TOTAL_COST|DIAGNOSTIC_CODE_COUNT|DIAGNOSTIC_CODES|NEXT_SERVICE_DATE|MAKE|MODEL|YEAR|FLEET_REGION|ODOMETER_ESTIMATE|DAYS_SINCE_LAST_SERVICE|SEVERITY_RANK|_change_type|_change_ordinal|_commit_snapshot_id|
+------+----------+-------------+----------+--------+-----------+----------+--------------+--------------+-----------+----------+---------------------+----------------+-----------------+----+-----+----+------------+-----------------+-----------------------+-------------+------------+---------------+-------------------+
+------+----------+-------------+----

In [40]:
spark.sql("""
SELECT operation, COUNT(*) cnt
FROM horizon.CURATED.TELEMETRY_ENRICHED.snapshots
GROUP BY operation
ORDER BY cnt DESC
""").show(truncate=False)

+---------+----+
|operation|cnt |
+---------+----+
|delete   |1817|
+---------+----+



In [42]:
from pyspark.sql import functions as F

catalog = "horizon"
namespaces = ["RAW", "CURATED", "ANALYTICS"]

ok = []
fail = []

for ns in namespaces:
    tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {catalog}.{ns}").collect()]
    for tbl in tables:
        full = f"{catalog}.{ns}.{tbl}"
        try:
            df = spark.sql(f"""
                SELECT
                    '{ns}'  AS namespace,
                    '{tbl}' AS table_name,
                    operation,
                    COUNT(*) AS snapshot_count
                FROM {full}.snapshots
                GROUP BY operation
            """)
            ok.append(df)
        except Exception as e:
            fail.append((ns, tbl, str(e)[:5000]))  # keep message bounded

# Union successes
if ok:
    out = ok[0]
    for d in ok[1:]:
        out = out.unionByName(d)
    out.orderBy("namespace", "table_name", F.desc("snapshot_count")).show(truncate=False)
else:
    print("No snapshot results collected.")

# Show failures
if fail:
    fail_df = spark.createDataFrame(fail, ["namespace", "table_name", "error"])
    fail_df.show(truncate=False)


+---------+------------------------+---------+--------------+
|namespace|table_name              |operation|snapshot_count|
+---------+------------------------+---------+--------------+
|ANALYTICS|DAILY_FLEET_SUMMARY     |delete   |463           |
|ANALYTICS|VEHICLE_HEALTH_SCORE    |delete   |246           |
|CURATED  |MAINTENANCE_ANALYSIS    |delete   |472           |
|CURATED  |TELEMETRY_ENRICHED      |delete   |1821          |
|RAW      |API_WEATHER_DATA        |append   |1             |
|RAW      |MAINTENANCE_LOGS        |append   |1             |
|RAW      |SENSOR_READINGS         |append   |1             |
|RAW      |VEHICLE_LOCATIONS       |append   |1             |
|RAW      |VEHICLE_TELEMETRY_STREAM|overwrite|1             |
+---------+------------------------+---------+--------------+



[Stage 59:>                                                         (0 + 1) / 1]

+---------+----------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [44]:
# Get latest snapshot + its parent
snaps = spark.sql("""
SELECT snapshot_id, parent_id, committed_at
FROM horizon.CURATED.TELEMETRY_ENRICHED.snapshots
ORDER BY committed_at DESC
LIMIT 1
""").collect()[0]

end_id = str(snaps["snapshot_id"])
start_id = snaps["parent_id"]

if start_id is None:
    raise ValueError("Latest snapshot has no parent (only one snapshot exists).")

start_id = str(start_id)

# Read bounded changelog window
df = (
    spark.read
        .option("start-snapshot-id", start_id)
        .option("end-snapshot-id", end_id)
        .table("horizon.CURATED.TELEMETRY_ENRICHED.changes")
)

# Show results properly
df.select("_change_type", "_commit_snapshot_id", "_change_ordinal") \
  .limit(20) \
  .show(20, truncate=False)


+------------+-------------------+---------------+
|_change_type|_commit_snapshot_id|_change_ordinal|
+------------+-------------------+---------------+
+------------+-------------------+---------------+



In [48]:
s = spark.sql("""
SELECT snapshot_id, parent_id, committed_at, operation
FROM horizon.PUBLIC.DEMO_CDC.snapshots
ORDER BY committed_at DESC
LIMIT 50
""")
s.show(truncate=False)
rows = s.collect()

+-------------------+-------------------+-----------------------+---------+
|snapshot_id        |parent_id          |committed_at           |operation|
+-------------------+-------------------+-----------------------+---------+
|2206959672966746869|8271189739922240429|2026-02-18 12:07:13.215|overwrite|
|8271189739922240429|7826053088527397858|2026-02-18 12:07:04.53 |overwrite|
|7826053088527397858|9176112333839631836|2026-02-18 12:06:57.171|overwrite|
|9176112333839631836|NULL               |2026-02-18 12:06:49.672|append   |
+-------------------+-------------------+-----------------------+---------+



In [50]:
end_id = str(rows[0]["snapshot_id"])
start_id = str(rows[2]["snapshot_id"])   # 10th back is guaranteed to be an ancestor in that list

df = (spark.read
        .option("start-snapshot-id", start_id)
        .option("end-snapshot-id", end_id)
        .table("horizon.PUBLIC.DEMO_CDC.changes"))

df.select("_change_type","_commit_snapshot_id","_change_ordinal").limit(20).show(20, truncate=False)


[Stage 68:>                                                         (0 + 1) / 1]

+------------+-------------------+---------------+
|_change_type|_commit_snapshot_id|_change_ordinal|
+------------+-------------------+---------------+
|INSERT      |2206959672966746869|1              |
|INSERT      |2206959672966746869|1              |
|INSERT      |2206959672966746869|1              |
|DELETE      |2206959672966746869|1              |
|DELETE      |2206959672966746869|1              |
|INSERT      |8271189739922240429|0              |
|INSERT      |8271189739922240429|0              |
|DELETE      |8271189739922240429|0              |
|DELETE      |8271189739922240429|0              |
|DELETE      |8271189739922240429|0              |
+------------+-------------------+---------------+



In [52]:
spark.sql("SELECT * FROM horizon.PUBLIC.DEMO_CDC.changes ORDER BY _change_ordinal DESC").show()

[Stage 70:>                                                         (0 + 1) / 1]

+---+---+------------+---------------+-------------------+
| ID|  V|_change_type|_change_ordinal|_commit_snapshot_id|
+---+---+------------+---------------+-------------------+
|  1|  a|      DELETE|              3|2206959672966746869|
|  2| b2|      DELETE|              3|2206959672966746869|
|  1|  a|      INSERT|              3|2206959672966746869|
|  2| b3|      INSERT|              3|2206959672966746869|
|  4|  d|      INSERT|              3|2206959672966746869|
|  1|  a|      DELETE|              2|8271189739922240429|
|  2| b2|      DELETE|              2|8271189739922240429|
|  3|  c|      DELETE|              2|8271189739922240429|
|  1|  a|      INSERT|              2|8271189739922240429|
|  2| b2|      INSERT|              2|8271189739922240429|
|  1|  a|      INSERT|              1|7826053088527397858|
|  2| b2|      INSERT|              1|7826053088527397858|
|  3|  c|      INSERT|              1|7826053088527397858|
|  1|  a|      DELETE|              1|782605308852739785

In [23]:
end_id = 9095944505780256261

spark.sql(f"""
SELECT committed_at, snapshot_id, parent_id, operation
FROM horizon.CURATED.TELEMETRY_ENRICHED.snapshots
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+
|committed_at           |snapshot_id        |parent_id          |operation|
+-----------------------+-------------------+-------------------+---------+
|2026-02-17 11:39:05.937|1690719396549071706|8598304144757295814|delete   |
|2026-02-17 11:39:54.514|8630857695058548190|1690719396549071706|delete   |
|2026-02-17 11:40:42.537|2955580276955871099|8630857695058548190|delete   |
|2026-02-17 11:41:30.043|8089555944130679286|2955580276955871099|delete   |
|2026-02-17 11:42:19.342|1731026451677640549|8089555944130679286|delete   |
|2026-02-17 11:43:07.592|2684670818739821125|1731026451677640549|delete   |
|2026-02-17 11:43:54.444|8521487258265308597|2684670818739821125|delete   |
|2026-02-17 11:44:42.449|7123094117137824809|8521487258265308597|delete   |
|2026-02-17 11:45:30.711|3838746889816527967|7123094117137824809|delete   |
|2026-02-17 11:46:19.708|747617706238703365 |3838746889816527967|delete   |
|2026-02-17 

In [26]:
spark.sql("""
SELECT snapshot_id, parent_id, committed_at, operation
FROM horizon.RAW.VEHICLE_TELEMETRY_STREAM.snapshots
ORDER BY committed_at DESC
LIMIT 50
""").show(truncate=False)

+-------------------+------------------+-----------------------+---------+
|snapshot_id        |parent_id         |committed_at           |operation|
+-------------------+------------------+-----------------------+---------+
|1007446037182973231|928296081656905320|2026-02-07 11:37:45.243|overwrite|
+-------------------+------------------+-----------------------+---------+



In [27]:
start_id = "928296081656905320"
end_id   = "1007446037182973231"

df = (spark.read
        .option("start-snapshot-id", start_id)
        .option("end-snapshot-id", end_id)
        .table("horizon.RAW.VEHICLE_TELEMETRY_STREAM.changes"))

df.select("VEHICLE_ID","EVENT_TIMESTAMP","_change_type","_commit_snapshot_id","_change_ordinal") \
  .limit(5).show(truncate=False)

26/02/18 11:45:13 ERROR BaseReader: Error reading file(s): s3://sfc-ca-ont-ds1-71-customer-interop-nfs-l8kf1000-s/30748836177/FLEET_ANALYTICS_DB/RAW/VEHICLE_TELEMETRY_STREAM.wT1CkeXE/data/77/snow_XJyoJkwZMkc_08Q_dvcNkhg_1_1_008.parquet
org.apache.iceberg.exceptions.NotFoundException: Location does not exist: s3://sfc-ca-ont-ds1-71-customer-interop-nfs-l8kf1000-s/30748836177/FLEET_ANALYTICS_DB/RAW/VEHICLE_TELEMETRY_STREAM.wT1CkeXE/data/77/snow_XJyoJkwZMkc_08Q_dvcNkhg_1_1_008.parquet
	at org.apache.iceberg.aws.s3.S3InputStream.openStream(S3InputStream.java:246)
	at org.apache.iceberg.aws.s3.S3InputStream.openStream(S3InputStream.java:229)
	at org.apache.iceberg.aws.s3.S3InputStream.positionStream(S3InputStream.java:225)
	at org.apache.iceberg.aws.s3.S3InputStream.read(S3InputStream.java:122)
	at org.apache.iceberg.shaded.org.apache.parquet.io.DelegatingSeekableInputStream.read(DelegatingSeekableInputStream.java:61)
	at org.apache.iceberg.shaded.org.apache.parquet.bytes.BytesUtils.readInt

Py4JJavaError: An error occurred while calling o235.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 8.0 failed 1 times, most recent failure: Lost task 0.0 in stage 8.0 (TID 8) (ip-10-0-0-167.us-west-2.compute.internal executor driver): org.apache.iceberg.exceptions.NotFoundException: Location does not exist: s3://sfc-ca-ont-ds1-71-customer-interop-nfs-l8kf1000-s/30748836177/FLEET_ANALYTICS_DB/RAW/VEHICLE_TELEMETRY_STREAM.wT1CkeXE/data/77/snow_XJyoJkwZMkc_08Q_dvcNkhg_1_1_008.parquet
	at org.apache.iceberg.aws.s3.S3InputStream.openStream(S3InputStream.java:246)
	at org.apache.iceberg.aws.s3.S3InputStream.openStream(S3InputStream.java:229)
	at org.apache.iceberg.aws.s3.S3InputStream.positionStream(S3InputStream.java:225)
	at org.apache.iceberg.aws.s3.S3InputStream.read(S3InputStream.java:122)
	at org.apache.iceberg.shaded.org.apache.parquet.io.DelegatingSeekableInputStream.read(DelegatingSeekableInputStream.java:61)
	at org.apache.iceberg.s

In [7]:
spark.sparkContext.uiWebUrl

'http://127.0.0.1:4040'

In [29]:
spark.sql("""
    SELECT
    VEHICLE_ID,
    _row_id,
    _last_updated_sequence_number
    FROM horizon.CURATED.TELEMETRY_ENRICHED
    WHERE
        _last_updated_sequence_number > 0
        AND _row_id <= 5
    ORDER BY _last_updated_sequence_number, _row_id
""").show()

26/02/18 10:25:26 WARN SparkScanBuilder: Failed to check if _last_updated_sequence_number IS NOT NULL can be pushed down: Cannot find field '_last_updated_sequence_number' in struct: struct<1: VEHICLE_ID: required string, 2: EVENT_TIMESTAMP: required timestamp, 3: TELEMETRY_DATA: required variant, 4: SPEED_MPH: optional double, 5: LATITUDE: optional double, 6: LONGITUDE: optional double, 7: HEADING: optional double, 8: ENGINE_RPM: optional decimal(38, 0), 9: ENGINE_TEMP_F: optional decimal(38, 0), 10: OIL_PRESSURE_PSI: optional double, 11: FUEL_LEVEL_PCT: optional double, 12: CHECK_ENGINE_LIGHT: optional boolean, 13: TIRE_PRESSURE_WARNING: optional boolean, 14: HARD_ACCELERATIONS: optional decimal(38, 0), 15: HARD_BRAKES: optional decimal(38, 0), 16: SHARP_TURNS: optional decimal(38, 0), 17: MAKE: optional string, 18: MODEL: optional string, 19: YEAR: optional decimal(10, 0), 20: DRIVER_ID: optional string, 21: DRIVER_NAME: optional string, 22: FLEET_REGION: optional string, 23: VEHICL

+----------+-------+-----------------------------+
|VEHICLE_ID|_row_id|_last_updated_sequence_number|
+----------+-------+-----------------------------+
|   VH-0019|      0|                            8|
|   VH-0011|      1|                            8|
|   VH-0011|      2|                            8|
|   VH-0012|      3|                            8|
|   VH-0004|      4|                            8|
|   VH-0011|      5|                            8|
+----------+-------+-----------------------------+



In [23]:
spark.sql("""
    SELECT
    VEHICLE_ID,
    _row_id,
    _last_updated_sequence_number
    FROM horizon.CURATED.TELEMETRY_ENRICHED VERSION AS OF 1336438565475268725
    WHERE
        _last_updated_sequence_number > 0
        AND _row_id <= 5
    ORDER BY _last_updated_sequence_number, _row_id
""").show()

26/02/18 10:19:12 WARN SparkScanBuilder: Failed to check if _last_updated_sequence_number IS NOT NULL can be pushed down: Cannot find field '_last_updated_sequence_number' in struct: struct<1: VEHICLE_ID: required string, 2: EVENT_TIMESTAMP: required timestamp, 3: TELEMETRY_DATA: required variant, 4: SPEED_MPH: optional double, 5: LATITUDE: optional double, 6: LONGITUDE: optional double, 7: HEADING: optional double, 8: ENGINE_RPM: optional decimal(38, 0), 9: ENGINE_TEMP_F: optional decimal(38, 0), 10: OIL_PRESSURE_PSI: optional double, 11: FUEL_LEVEL_PCT: optional double, 12: CHECK_ENGINE_LIGHT: optional boolean, 13: TIRE_PRESSURE_WARNING: optional boolean, 14: HARD_ACCELERATIONS: optional decimal(38, 0), 15: HARD_BRAKES: optional decimal(38, 0), 16: SHARP_TURNS: optional decimal(38, 0), 17: MAKE: optional string, 18: MODEL: optional string, 19: YEAR: optional decimal(10, 0), 20: DRIVER_ID: optional string, 21: DRIVER_NAME: optional string, 22: FLEET_REGION: optional string, 23: VEHICL

+----------+-------+-----------------------------+
|VEHICLE_ID|_row_id|_last_updated_sequence_number|
+----------+-------+-----------------------------+
|   VH-0019|      0|                            8|
|   VH-0011|      1|                            8|
|   VH-0011|      2|                            8|
|   VH-0012|      3|                            8|
|   VH-0004|      4|                            8|
|   VH-0011|      5|                            8|
+----------+-------+-----------------------------+



In [28]:
CALL spark_catalog.system.create_changelog_view(
  table => 'horizon.CURATED.TELEMETRY_ENRICHED',
  options => map('1336438565475268725','1','6001879231150925298', '2')
);

SyntaxError: invalid syntax (847364944.py, line 1)

In [20]:
spark.sql("""
SELECT 
    *
FROM horizon.CURATED.TELEMETRY_ENRICHED.history
""").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-02-17 08:43:53.845|5620748715537758310|7094277992954599874|true               |
|2026-02-17 08:44:41.824|5439893651973053702|5620748715537758310|true               |
|2026-02-17 08:45:29.919|7400554709204656485|5439893651973053702|true               |
|2026-02-17 08:46:17.371|435646511617153993 |7400554709204656485|true               |
|2026-02-17 08:47:05.851|2274585298479016826|435646511617153993 |true               |
|2026-02-17 08:47:53.792|2440826552977439020|2274585298479016826|true               |
|2026-02-17 08:48:42.244|6107182849100608830|2440826552977439020|true               |
|2026-02-17 08:49:29.485|4560427282162804655|6107182849100608830|true               |
|2026-02-17 08:50:17.477|2979426286250673734|456042728

In [ ]:
# See variant column in Iceberg table
spark.sql("DESCRIBE TABLE horizon.RAW.VEHICLE_TELEMETRY_STREAM").show(truncate=False)

## Query Data and Extract VARIANT Fields

In [ ]:
# Use Spark's variant_get() function to extract nested fields from VARIANT
# This query succeeds, and no Snowflake compute is used, since there's no masking policy on the Iceberg table.
df = spark.sql(f"""
    SELECT 
        VEHICLE_ID,
        EVENT_TIMESTAMP,
        variant_get(TELEMETRY_DATA, '$.speed_mph', 'float') AS speed_mph,
        variant_get(TELEMETRY_DATA, '$.engine.temperature_f', 'int') AS engine_temp,
        variant_get(TELEMETRY_DATA, '$.location.lat', 'float') AS latitude,
        variant_get(TELEMETRY_DATA, '$.location.lon', 'float') AS longitude
    FROM horizon.RAW.VEHICLE_TELEMETRY_STREAM
    WHERE variant_get(TELEMETRY_DATA, '$.speed_mph', 'float') > 60
    LIMIT 10
""")
df.show()

## Demonstrate Access Control (Masking Enforcement)

In [ ]:
# To enforce masking policies from Spark, we need the Snowflake Connector for Spark
# which routes queries through Snowflake for policy evaluation
# See: https://docs.snowflake.com/en/user-guide/tables-iceberg-query-using-external-query-engine-snowflake-horizon-enforce-access-policies

# Stop the previous Spark session first
spark.stop()
from pyspark.sql import SparkSession

# Configuration
SNOWFLAKE_PASSWORD = os.getenv('SNOWFLAKE_PASSWORD')
SNOWFLAKE_ROLE = 'FLEET_ANALYST'
SNOWFLAKE_SCHEMA = 'RAW'
SNOWFLAKE_WAREHOUSE = os.getenv('SNOWFLAKE_WAREHOUSE', 'FLEET_ANALYTICS_WH')
SF_URL = f"{SNOWFLAKE_ACCOUNT_URL}.snowflakecomputing.com"

# Versions - Note: Spark 3.5 required for Snowflake Connector masking support
SNOWFLAKE_JDBC_VERSION = "3.24.0"
SNOWFLAKE_SPARK_CONNECTOR_VERSION = "3.1.6"

spark_analyst = SparkSession.builder \
    .appName("Fleet Analytics - Analyst View") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.jars.packages", 
            f"org.apache.iceberg:iceberg-spark-runtime-4.0_{SCALA_VERSION}:{ICEBERG_VERSION},"
            f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION},"
            f"net.snowflake:snowflake-jdbc:{SNOWFLAKE_JDBC_VERSION},"
            f"net.snowflake:spark-snowflake_{SCALA_VERSION}:{SNOWFLAKE_SPARK_CONNECTOR_VERSION}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.defaultCatalog", "horizon") \
    .config("spark.sql.catalog.horizon", "org.apache.spark.sql.snowflake.catalog.SnowflakeFallbackCatalog") \
    # .config("spark.sql.catalog.horizon.catalog-impl", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.horizon.type", "rest") \
    .config("spark.sql.catalog.horizon.uri", f"https://{SNOWFLAKE_ACCOUNT_URL}.snowflakecomputing.com/polaris/api/catalog") \
    .config("spark.sql.catalog.horizon.warehouse", SNOWFLAKE_DATABASE) \
    .config("spark.sql.catalog.horizon.scope", f"session:role:{SNOWFLAKE_ROLE}") \
    .config("spark.sql.catalog.horizon.credential", SNOWFLAKE_FLEET_ANALYST_TOKEN) \
    .config("spark.sql.catalog.horizon.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.horizon.header.X-Iceberg-Access-Delegation", "vended-credentials") \
    .config("spark.snowflake.sfURL", SF_URL) \
    .config("spark.snowflake.sfUser", os.getenv('SNOWFLAKE_USER')) \
    .config("spark.snowflake.sfPassword", SNOWFLAKE_FLEET_ANALYST_TOKEN) \
    .config("spark.snowflake.sfDatabase", SNOWFLAKE_DATABASE) \
    .config("spark.snowflake.sfSchema", SNOWFLAKE_SCHEMA) \
    .config("spark.snowflake.sfRole", SNOWFLAKE_ROLE) \
    .config("spark.snowflake.sfWarehouse", SNOWFLAKE_WAREHOUSE) \
    .config("spark.sql.iceberg.vectorization.enabled", "false") \
    .getOrCreate()

spark_analyst.sparkContext.setLogLevel("ERROR")

In [ ]:
# Still see Iceberg tables via Iceberg REST API
spark_analyst.sql("SHOW TABLES in horizon.RAW").show(truncate=False)

In [ ]:
# When reading an Iceberg table with a maskign policy, Snowflake returns masked results
spark_analyst.sql("""
    SELECT
        VEHICLE_ID,
        MAKE,
        MODEL,
        YEAR,
        LICENSE_PLATE,
        DRIVER_NAME,
        DRIVER_EMAIL,
        DRIVER_PHONE,
        FLEET_REGION
    FROM horizon.RAW.VEHICLE_REGISTRY
""").show(truncate=True)

## Summary

This notebook demonstrated:
1. **Connecting Spark to Snowflake Iceberg Catalog** using REST API
2. **Listing and describing tables** including Dynamic Iceberg Tables
3. **Querying VARIANT data** with JSON path extraction
4. **Access control enforcement** with masking policies applied in Spark